In [ ]:
import cv2
import numpy as np

def image_slice(frame:np.ndarray, y1, y2, x1, x2): #自由度是4，因為有y1, y2, x1, x2來決定一個長方形、斜線也是
    height,width,_=frame.shape #或height,width=frame.shape[:2]
                  #shape前兩個一定是高寬，後面的不管(原height,width,)→用[:2]
    return frame[int(y1*height):int(y2*height), int(x1*width):int(x2*width)]


def draw_rectangle(frame:np.ndarray, y1, y2, x1, x2):
    height,width,_=frame.shape


    cv2.rectangle(       #opencv的套件
        frame,
        (int(x1*width), int( y1*height)),
        (int(x2*width), int( y2*height)),
        (0, 0, 0),   #畫的框的顏色
        2         #畫的框的粗細
    )
    return frame


def c(frame:np.ndarray,fast=True):  #旋轉畫面功能np flip vertical
    if fast==True:   #向量化、用 NumPy 內建函數快速旋轉
        return np.flip(np.permute_dims(frame,[1,0,2]), 0) #flip是上下翻轉

    else:   #慢速、用兩層 for 迴圈逐 pixel 計算旋轉
        height,width,_=frame.shape
        e90=[]
        for i in range(width):
            e90.append([])
            for j in range(height):
                e90[-1].append(frame[j,width-1-i])
        return np.array(e90)


cap=cv2.VideoCapture(0)
if cap.isOpened():
    ret,frame=cap.read() #frame 是：1.一張攝影機即時畫面 2.型別：numpy.ndarray 3.內容：像素矩陣 (H, W, 3)
    rot90flag=False    #用來作旋轉的控制器
    drawrectflag=False #用來作畫框的控制器
    sliceflag = False  #用來作切片的控制器

    while ret:

        #處理鍵盤事件
        key=cv2.waitKey(1)    #等一毫秒沒反應就下一個
        if key==ord['r']: rot90flag=not rot90flag #按r建旋轉
        elif key==ord['q']: break
        elif key==ord['s']: sliceflag=not sliceflag   #切片
        elif key==ord['e']: drawrectflag=not drawrectflag   #畫方形flag

        #依據flag狀態做影像處理
        if rot90flag: frame=rotate90ccw(frame)  #用新影像取代原本的 frame
        if sliceflag: frame=image_slice(frame, 0.2, 0.9, 0.1, 0.5)
        if drawrectflag: frame=draw_rectangle(frame, 0.2, 0.9, 0.1, 0.5)
        cv2.imshow('Hi',frame)
        #讀取flag狀態做影像處理
        ret,frame=cap.read()


cap.release()
cv2.destroyAllWindows()
